# RDP HTTPX — Synchronous Historical Interday Summaries

Demonstrates a complete end-to-end request flow against [LSEG Data Platform (RDP)](https://developers.lseg.com/en/api-catalog/refinitiv-data-platform/refinitiv-data-platform-apis) using [`httpx`](https://www.python-httpx.org/) in **synchronous** mode with a shared `httpx.Client`:

1. Authenticate with OAuth 2.0 Password Grant to obtain a Bearer token
2. Fetch historical interday price summaries (daily OHLCV) for a list of RICs
3. Revoke the session token on completion

> **Note:** `verify=False` disables TLS certificate verification — intended for local/dev environments only (e.g. behind a ZScaler proxy). Remove it for production use.

In [10]:
import os
import time
import json
import httpx
from dotenv import load_dotenv

## Configuration

API endpoint paths and the RIC universe are defined as constants. Credentials (`MACHINEID_RDP`, `PASSWORD_RDP`, `APPKEY_RDP`, `RDP_BASE_URL`) are loaded from `src/.env` via `python-dotenv`.

In [11]:
AUTH_TOKEN_URL = "/auth/oauth2/v1/token"
AUTH_REVOKE_URL = "/auth/oauth2/v1/revoke"
HISTORICAL_INTERDAY_SUMMARIES_URL = "/data/historical-pricing/v1/views/interday-summaries/"
HISTORICAL_RICS = ["AAPL.O","MSFT.O","META.O","NVDA.O","GOOG.O","ORCL.N","IBM.N","PLTR.O","AMZN.O","AVGO.O"]  # Fetched concurrently

In [12]:
load_dotenv()

# Read credentials and base URL from environment variables.
machine_id = os.getenv("MACHINEID_RDP")
password = os.getenv("PASSWORD_RDP")
app_key = os.getenv("APPKEY_RDP")
base_url = os.getenv("RDP_BASE_URL")

## Helper Functions

Three reusable functions used by the main execution block. All accept the shared `httpx.Client` instance so the same connection pool is reused across every call.

| Function | Purpose |
|---|---|
| `post_authentication` | OAuth 2.0 Password Grant — returns a Bearer token |
| `post_auth_revoke` | Invalidates the session token when done |
| `get_historical_interday_summaries` | Fetches daily OHLCV data for a single RIC |

In [13]:
def post_authentication(machine_id, password, app_key, url, client):
    """Authenticate to RDP and return the token response as JSON."""

    # Build the OAuth 2.0 Password Grant request payload.
    # Sent as application/x-www-form-urlencoded (httpx encodes a dict automatically).
    payload = {
        "username": machine_id,           # RDP Machine-ID
        "password": password,             # RDP Password
        "grant_type": "password",         # OAuth 2.0 grant type
        "scope": "trapi",                 # Target API scope
        "takeExclusiveSignOnControl": "true",  # Revoke other active sessions
        "client_id": app_key              # RDP AppKey (acts as client_id)
    }

    # Send authentication request to the OAuth token endpoint.
    # `data=payload` sends a form body required by this endpoint.
    response = client.post(url, data=payload)
    response.raise_for_status()  # Raise for 4xx/5xx API failures.
    return response.json()

In [14]:
def post_auth_revoke(token, app_key, url, client):
    """Revoke the access token to end the session."""
    headers = {
        "Content-Type": "application/x-www-form-urlencoded"
    }

    payload = f"token={token}"
    auth = httpx.BasicAuth(username=app_key, password="")
    response = client.post(url, data=payload, headers=headers, auth=auth)
    response.raise_for_status()

In [15]:
def get_historical_interday_summaries(ric, token, url, client, interval, start, end, fields):
    """Request historical Interday summaries data for a single RIC.
    """
    print(f"Fetching historical interday summaries... for RIC: {ric}")

    # Bearer token authenticates the caller; Content-Type signals a JSON response is expected.
    headers = {
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json"
    }

    # Build the query string for the interday-summaries endpoint.
    # adjustments: apply standard corporate-action and price corrections so that
    # the returned series is comparable across the full date range.
    # fields: comma-separated list of data columns to include in the response.
    params = {
        "interval": interval,
        "start": start,
        "end": end,
        "adjustments": "exchangeCorrection,manualCorrection,CCH,CRE,RPO,RTS",
        "fields": ",".join(fields)
    }

    response_history =  client.get(f"{url}{ric}", params=params, headers=headers)

    print(f"Request URL: {response_history.url}")

    # Raise an exception for 4xx/5xx HTTP errors; lets the caller handle
    # status-specific logic (e.g. 429 rate-limit vs. 401 auth failure).
    response_history.raise_for_status()

    # Deserialise and return the JSON payload for further processing by the caller.
    return response_history.json()

## Main Execution

Start the wall-clock timer, open a single shared `httpx.Client`, then run the full workflow: **authenticate → fetch data → revoke token**. The `with` block guarantees the client and its connection pool are closed cleanly regardless of success or error.

In [16]:
start_time = time.perf_counter()

### Open a shared `httpx.Client`

Using a `with` block ensures the connection pool is closed (and open connections flushed) when the block exits — whether it completes normally or raises an exception. All three API calls share the same underlying TCP connections, reducing overhead compared to creating a new client per request.

In [17]:
with httpx.Client(
    verify=False,
    base_url=base_url,
    timeout=10.0,
    default_encoding="utf-8",
    follow_redirects=True,
) as client:
    try:
        # Authenticate and get the access token.
        auth_response = post_authentication(machine_id, password, app_key, AUTH_TOKEN_URL, client)
        access_token = auth_response["access_token"]
        print("Authentication successful! Access token obtained.\n")

        # Fetch historical interday summaries for each RIC sequentially.
        # For better performance with many RICs, consider concurrent requests (e.g. using asyncio or threading).
        fields = ["TRDPRC_1", "BID", "ASK"]
        start_date = "2025-11-01T00:00:00Z"
        end_date = "2026-02-28T23:59:59Z"
        for ric in HISTORICAL_RICS:
            history_data =  get_historical_interday_summaries(
                ric, access_token, HISTORICAL_INTERDAY_SUMMARIES_URL, client, interval="P1D", start=start_date, end=end_date, fields=fields
            )
            print("Historical interday summaries retrieved successfully!")
            print(f"Historical interday summaries for '{ric}': {history_data}\n\n")
        
        time.sleep(1)  # Sleep briefly to ensure all requests are completed before revoking the token.
        print("Revoking access token...")
        post_auth_revoke(access_token, app_key, AUTH_REVOKE_URL, client)
        print("Access token revoked successfully.\n")
    except httpx.HTTPStatusError as exc:
        print(f"HTTP error occurred during HTTP Request: {exc.request.url}: {exc.response.status_code} - {exc.response.text}")
    except httpx.RequestError as exc:
        print(f"An error occurred during HTTP Request: {exc.request.url}: {exc}")
    except ValueError as exc:
        print(f"Configuration error: {exc}")
    except KeyError as exc:
        print(f"An error occurred during HTTP Request: {exc}")

Authentication successful! Access token obtained.

Fetching historical interday summaries... for RIC: AAPL.O
Request URL: https://api.refinitiv.com/data/historical-pricing/v1/views/interday-summaries/AAPL.O?interval=P1D&start=2025-11-01T00%3A00%3A00Z&end=2026-02-28T23%3A59%3A59Z&adjustments=exchangeCorrection%2CmanualCorrection%2CCCH%2CCRE%2CRPO%2CRTS&fields=TRDPRC_1%2CBID%2CASK
Historical interday summaries retrieved successfully!
Historical interday summaries for 'AAPL.O': [{'universe': {'ric': 'AAPL.O'}, 'interval': 'P1D', 'summaryTimestampLabel': 'endPeriod', 'adjustments': ['exchangeCorrection', 'manualCorrection', 'CCH', 'CRE', 'RTS', 'RPO'], 'defaultPricingField': 'TRDPRC_1', 'headers': [{'name': 'DATE', 'type': 'string'}, {'name': 'TRDPRC_1', 'type': 'number', 'decimalChar': '.'}, {'name': 'BID', 'type': 'number', 'decimalChar': '.'}, {'name': 'ASK', 'type': 'number', 'decimalChar': '.'}], 'data': [['2026-02-27', 264.18, 264.17, 264.21], ['2026-02-26', 272.95, 272.99, 273.01], 

In [18]:
elapsed = time.perf_counter() - start_time
print(f"simple_call_nb.ipynb executed for {HISTORICAL_RICS} in {elapsed:0.2f} seconds.")

simple_call_nb.ipynb executed for ['AAPL.O', 'MSFT.O', 'META.O', 'NVDA.O', 'GOOG.O', 'ORCL.N', 'IBM.N', 'PLTR.O', 'AMZN.O', 'AVGO.O'] in 9.94 seconds.
